# Comparing Advanced Architectures: ViT and ConvNeXt V2 (Actuator-Aligned Patches)

`Tutorials/Advanced/CNNArchitectureComparison.ipynb` benchmarked six small, plain/grouped-conv reconstructors against each other, deliberately staying at "small CNN, single-GPU" scale. This notebook makes the transformer/ConvNeXt-scale departure that one explicitly avoided: it benchmarks a Vision Transformer (**ViT**) and a **ConvNeXt V2** against a classic-CNN baseline (`ClassicCNN`, copied verbatim from that notebook), trained and evaluated with the exact same instrument, loss, and `AI4AO.Trainer.Trainer` procedure. See `Ideas/11-advanced-architecture-comparison.md` for the full design discussion and the Q&A that resolved its open forks.

**The actuator-aligned patch grid.** ViT and ConvNeXt V2 are both patch-based architectures. Rather than an arbitrary vision-model patch size, this notebook patchifies each WFS pupil image on a grid sized to the DM's own actuator grid (`PATCHES_PER_SIDE`, a single constant set in the configuration cell) -- so each patch/token has roughly the same physical footprint as one actuator's zone of influence, the part of the pupil that actuator is actually responsible for correcting. Note that `PATCHES_PER_SIDE` is pinned at 11 (the 11-actuator DM this notebook was first built for) rather than following `DMParams["Nactuator"]`: the shared `CNNComparison_params.py` now describes a 17-actuator DM, and a 17x17 grid would make the ViT's closed-loop training stage need ~34 GiB of GPU memory, so with the shared params one patch spans roughly 1.5 actuators.

**Three architectures, one shared `PatchEmbed` stem**, each processing the 4 registered pupil crops *independently* (their own patch grid per pupil, concatenated afterward) rather than retiling them into one image first:

1. **`ClassicCNN`** -- the same plain, non-grouped `Conv2d` stack baseline as the six-architecture notebook. No patches, no positional information -- included here as the un-patched reference point.
2. **`ViT`** -- patchify stem (shared `PatchEmbed`, `groups=4`) -> per-patch positional embedding (from each patch's normalized pupil-plane coordinates) + a learned per-pupil identity embedding (since self-attention has no inherent notion of location or pupil identity) -> pre-norm Transformer encoder blocks -> mean-pooled head.
3. **`ConvNeXtV2`** -- the same `PatchEmbed` stem, kept as a 2-D feature map instead of a token sequence (convolution gives it locality for free, so no explicit position embedding is needed) -> ConvNeXt V2 stages (depthwise conv + a from-scratch Global Response Normalization block, the actual V2-over-V1 ingredient) -> pooled head.

All three are sized to land close to the same ~3M-trainable-parameter budget (the `3M_wide` size of `WidthDepthComparison.ipynb`; the six-architecture notebook used ~0.9M), so the comparison isolates architecture rather than capacity. All three train with the same `AdamW` optimizer and learning rate -- a deliberate choice to keep architecture the only controlled variable, at the cost of possibly understating what ViT/ConvNeXt V2 could reach under their own typically-recommended (schedule-sensitive) recipes. See the "Known risks" discussion in `Ideas/11-advanced-architecture-comparison.md` for the full reasoning, in particular the `O(N^2)` cost of self-attention over `n_pupils * patches_per_side^2` tokens (484 at the current default) -- this is the concrete form the "not a transformer-scale departure" caution from idea 07 takes here.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

from AI4AO.paths import DATA_DIR
from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow_multiple
from AI4AO.LossFunctions import LogResidualVarianceLoss, Physics_loss

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Configuration and instrument

Same fictional-demo, nominal-geometry modulated Pyramid WFS + DM as `CNNArchitectureComparison.ipynb` -- this notebook imports `CNNComparison_params.py` directly rather than forking its own copy (per the Q&A in `Ideas/11-advanced-architecture-comparison.md`), so both notebooks always compare architectures on the identical instrument. `wfs`/`dm` are built fresh and frozen (`.eval()`); only each architecture's reconstructor weights are trained.

`PATCHES_PER_SIDE` is the one constant that controls every architecture's patch grid below -- every architecture takes it as a constructor argument, so changing it here is the only edit needed to try a different patch resolution.

In [ ]:
paramfile = 'CNNComparison_params.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

PATH = DATA_DIR / "AdvancedArchitectureComparison"
os.makedirs(PATH, exist_ok=True)

wfs = PyramidWFS(WFSParams, device)
wfs.eval()

dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()

framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

M2C = dm.MakeZernikeM2C()
z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))

loss = LogResidualVarianceLoss(dataset.pupil, wfs.wavelength) + Physics_loss(wfs=wfs)

n_channels = wfs.pupil_centers.shape[0]
Nout = framePreprocessor.Nout
Nmodes = DMParams["Nmodes"]

# Patch rows/columns per pupil for ViT and ConvNeXtV2. Not in CNNComparison_params.py (it only
# applies to this notebook), so it lives here. Pinned to 11 instead of tying it to
# DMParams["Nactuator"] (17 in the shared params): a 17x17 grid gives the ViT 289 tokens per
# pupil (1,156 in total) and self-attention memory grows with the square of that. Measured on
# an 8 GB laptop GPU, stage-2 training (10 closed-loop iterations) of the earlier ~0.9M-parameter
# ViT needed ~34 GiB at 17x17 vs ~7.7 GiB at 11x11; the current ~3M-parameter ViT at 11x11
# needs ~10 GiB (~1.3 GiB at 1 iteration, +~1 GiB per extra iteration).
# PatchEmbed pads Nout up to an exact multiple of this value internally, so any positive
# integer works, not just ones that divide Nout evenly.
PATCHES_PER_SIDE = 11

print(f"Pyramid pupils: {n_channels}, preprocessed pupil size: {Nout}x{Nout}, "
      f"Nmodes: {Nmodes}, patches per side: {PATCHES_PER_SIDE}")

## `PatchEmbed` -- the shared actuator-aligned patch stem

Patchifies each of the `n_channels` pupil crops *independently* via one `groups=n_channels` `Conv2d`, the same idiom `StemEncoderCNN`'s stem uses in the six-architecture notebook -- each pupil gets its own kernel weights and contributes its own, fixed slice of the output channels (pupil `k`'s features occupy channels `[k*embed_dim_per_pupil : (k+1)*embed_dim_per_pupil]`), so pupil identity is preserved by construction rather than by convention.

`Nout` (42 at the current params) is not generally divisible by `PATCHES_PER_SIDE` (11, pinned in the configuration cell), so the input is zero-padded symmetrically up to `PATCHES_PER_SIDE * patch_pixels` before the strided conv -- `PATCHES_PER_SIDE` is honored exactly for any `Nout`, not only ones that divide evenly. With the current setting that pads 42 up to 44 pixels (2 pixels split across each side), so the outermost ring of patches is partly zero-padding -- worth seeing by eye once, which the next cell does alongside a numerical check that no pupil's patches leak information from another pupil.

In [ ]:
class PatchEmbed(nn.Module):
    """Patchifies n_channels pupil crops (B, n_channels, Nout, Nout) independently via one
    groups=n_channels Conv2d, producing an (n_channels * embed_dim_per_pupil)-channel
    (patches_per_side, patches_per_side) feature map. Pupil k's features occupy channels
    [k*embed_dim_per_pupil : (k+1)*embed_dim_per_pupil] -- a fixed, known slice, not an
    incidental one, since groups=n_channels gives each pupil its own kernel weights and its
    own contiguous output-channel block.

    Nout need not be divisible by patches_per_side: the input is zero-padded symmetrically
    up to patches_per_side * patch_pixels before the strided conv, so patches_per_side is
    always honored exactly."""

    def __init__(self, Nout, n_channels, embed_dim_per_pupil, patches_per_side):
        super().__init__()
        self.n_channels = n_channels
        self.embed_dim_per_pupil = embed_dim_per_pupil
        self.patches_per_side = patches_per_side

        self.patch_pixels = -(-Nout // patches_per_side)  # ceil division
        pad_total = self.patch_pixels * patches_per_side - Nout
        pad_before = pad_total // 2
        pad_after = pad_total - pad_before
        self.pad = (pad_before, pad_after, pad_before, pad_after)  # (left, right, top, bottom)

        self.proj = nn.Conv2d(
            n_channels, n_channels * embed_dim_per_pupil,
            kernel_size=self.patch_pixels, stride=self.patch_pixels, groups=n_channels,
        )

    def forward(self, x):
        if any(self.pad):
            x = F.pad(x, self.pad)
        return self.proj(x)  # (B, n_channels*embed_dim_per_pupil, patches_per_side, patches_per_side)

In [ ]:
# Sanity check: perturbing only one pupil's input must only change that pupil's channel
# block in PatchEmbed's output. This is the same class of silent-failure risk
# CNNArchitectureComparison.ipynb's retile_pupils validates by eye -- here it's checked
# numerically (and is cheap enough to leave in as a standing regression check).
_check_embed = PatchEmbed(Nout, n_channels, embed_dim_per_pupil=6, patches_per_side=PATCHES_PER_SIDE).to(device)
with torch.no_grad():
    x1 = torch.randn(1, n_channels, Nout, Nout, device=device)
    x2 = x1.clone()
    x2[:, 1] = torch.randn(Nout, Nout, device=device)  # perturb only pupil index 1

    diff = (_check_embed(x1) - _check_embed(x2)).abs()
    C = _check_embed.embed_dim_per_pupil
    for k in range(n_channels):
        max_diff = diff[:, k * C:(k + 1) * C].max().item()
        assert (k == 1) == (max_diff > 0), (
            f"PatchEmbed leaked information across pupils (pupil block {k}, max diff {max_diff})"
        )
print("PatchEmbed pupil isolation check passed.")
print(f"patch_pixels={_check_embed.patch_pixels}, pad={_check_embed.pad} "
      f"(padded size {PATCHES_PER_SIDE * _check_embed.patch_pixels} vs Nout={Nout})")

# Eyeball check: one preprocessed pupil crop with the patch grid overlaid, so the patch
# footprint relative to the pupil is visible before trusting any result below.
with torch.no_grad():
    batch = dataset[0]
    wfs.SetPhotonsAndRON(batch["nphotons"], batch["ron"])
    check_frames = wfs(batch["opd"], batch["pupil"])
    check_preprocessed = framePreprocessor.ProcessFrame(check_frames, add_pupil_noise=False)

pad_before = _check_embed.pad[0]
patch_pixels = _check_embed.patch_pixels

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(check_preprocessed[0, 0].cpu(), cmap="viridis")
for i in range(PATCHES_PER_SIDE + 1):
    coord = i * patch_pixels - pad_before - 0.5
    ax.axhline(coord, color="white", linewidth=0.5, alpha=0.7)
    ax.axvline(coord, color="white", linewidth=0.5, alpha=0.7)
ax.set_xlim(0, Nout - 1)
ax.set_ylim(Nout - 1, 0)
ax.set_title(f"Pupil 0 with {PATCHES_PER_SIDE}x{PATCHES_PER_SIDE} actuator-aligned patch grid")
plt.show()

## Baseline: `ClassicCNN`

Copied verbatim from `CNNArchitectureComparison.ipynb` (including its `num_downsamples`/`conv_stage` helpers) so the baseline here is a literal match to that notebook's own "channels are just channels" architecture, rather than a re-invented one -- no patches, no positional information. It is instantiated below at `base_channels=51`, the `3M_wide` network of `WidthDepthComparison.ipynb` (~3.05M parameters at `Nmodes=190`), instead of that notebook's default `base_channels=28`, so the baseline sits at the same ~3M budget as the other two.

In [ ]:
def num_downsamples(size, min_size=2):
    """How many stride-2 poolings a feature map of this spatial size can take before
    dropping below min_size pixels -- lets each architecture below downsample as deep as
    the actual resolution allows, rather than a fixed, possibly-too-shallow depth."""
    n = 0
    while size // 2 >= min_size:
        size //= 2
        n += 1
    return n


def conv_stage(in_channels, out_channels, activation=nn.LeakyReLU):
    """Two 3x3 convs at out_channels, then a 2x downsampling MaxPool."""
    return [
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        activation(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        activation(),
        nn.MaxPool2d(2),
    ]


class ClassicCNN(nn.Module):
    """Plain, non-grouped Conv2d stack -- channels are just channels, no per-pupil
    structure and no positional information. Identical to CNNArchitectureComparison.ipynb's
    ClassicCNN."""

    def __init__(self, n_channels, Nmodes, Nout, base_channels=28):
        super().__init__()

        n_stages = num_downsamples(Nout)
        channels = [n_channels] + [base_channels * 2 ** i for i in range(n_stages)]

        layers = []
        for i in range(n_stages):
            layers += conv_stage(channels[i], channels[i + 1])
        layers.append(nn.AdaptiveAvgPool2d(1))

        self.encoder = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(channels[-1], Nmodes))

    def forward(self, x):
        x = self.encoder(x)
        return self.head(x)

## `ViT` -- actuator-aligned patches, per-pupil token sequence

`PatchEmbed`'s stem output `(B, n_channels*embed_dim_per_pupil, P, P)` is reshaped into a token sequence: pupil `k`'s `embed_dim_per_pupil`-channel block becomes its own `P*P` tokens, and the four pupils' token sets are concatenated along the sequence dimension (`n_channels*P*P` tokens total -- 484 at the current defaults) -- this is what "per-pupil patchify" means concretely, as opposed to retiling the four pupils into one image before patchifying.

Self-attention has no inherent notion of location, so two embeddings are added to every token before the Transformer blocks:

- a **positional embedding** built from each patch's normalized `(x, y)` center in the pupil plane (the same `linspace(-1, 1, ...)` convention `CoordConvCNN` uses in the six-architecture notebook, at patch resolution instead of pixel resolution), projected to `embed_dim_per_pupil` by a small `Linear`;
- a learned **pupil-identity embedding**, one `embed_dim_per_pupil`-vector per pupil -- needed because after concatenation, tokens from different pupils otherwise share the exact same normalized-coordinate range and would be indistinguishable to attention.

Standard pre-norm Transformer encoder blocks follow (`LayerNorm` -> multi-head self-attention -> residual; `LayerNorm` -> 2-layer GELU MLP -> residual), then a mean-pool over all tokens and a `Linear` head -- no `CLS` token, keeping the same pooled-feature-then-`Linear` convention every other architecture here and in the six-architecture notebook ends with.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, n_heads, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

    def forward(self, x):
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed, need_weights=False)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class ViT(nn.Module):
    """Actuator-aligned-patch Vision Transformer. Each pupil is patchified independently
    (shared PatchEmbed, groups=n_channels) into its own P*P tokens; the n_channels token
    sets are concatenated, given a per-patch positional embedding and a per-pupil identity
    embedding, then processed by `depth` pre-norm Transformer blocks."""

    def __init__(self, Nout, n_channels, Nmodes, patches_per_side,
                 embed_dim_per_pupil=248, depth=4, n_heads=8, mlp_ratio=4.0):
        super().__init__()
        self.n_channels = n_channels
        self.patches_per_side = patches_per_side
        self.embed_dim_per_pupil = embed_dim_per_pupil

        self.patch_embed = PatchEmbed(Nout, n_channels, embed_dim_per_pupil, patches_per_side)

        yy, xx = torch.meshgrid(
            torch.linspace(-1, 1, patches_per_side), torch.linspace(-1, 1, patches_per_side),
            indexing="ij",
        )
        coords = torch.stack([xx, yy], dim=-1).reshape(patches_per_side * patches_per_side, 2)
        self.register_buffer("patch_coords", coords)  # (P*P, 2) -- fixed geometry, not learned
        self.pos_proj = nn.Linear(2, embed_dim_per_pupil)
        self.pupil_embed = nn.Parameter(torch.zeros(n_channels, embed_dim_per_pupil))

        self.blocks = nn.ModuleList(
            [TransformerBlock(embed_dim_per_pupil, n_heads, mlp_ratio) for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(embed_dim_per_pupil)
        self.head = nn.Linear(embed_dim_per_pupil, Nmodes)

    def forward(self, x):
        feat = self.patch_embed(x)  # (B, n_channels*C, P, P)
        B = feat.shape[0]
        P = self.patches_per_side
        C = self.embed_dim_per_pupil

        feat = feat.view(B, self.n_channels, C, P, P)
        tokens = feat.flatten(3).permute(0, 1, 3, 2)  # (B, n_channels, P*P, C)

        pos = self.pos_proj(self.patch_coords)  # (P*P, C)
        tokens = tokens + pos.view(1, 1, P * P, C) + self.pupil_embed.view(1, self.n_channels, 1, C)
        tokens = tokens.reshape(B, self.n_channels * P * P, C)

        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.norm(tokens)

        pooled = tokens.mean(dim=1)
        return self.head(pooled)

## `ConvNeXtV2` -- actuator-aligned patches, convolutional stages

Uses the same `PatchEmbed` stem as `ViT` (shared module, shared `PATCHES_PER_SIDE`), but its output is kept as a 2-D feature map rather than flattened into tokens: convolution has locality and the per-pupil channel grouping built in, so unlike `ViT`, no explicit positional or pupil-identity embedding is needed here -- each block's depthwise convolution stays local (and, in the first stage, stays within one pupil's channel block), while cross-pupil fusion happens exactly where the pointwise 1x1 layers are, the same place `StemEncoderCNN`'s shared encoder mixes across pupils in the six-architecture notebook.

The number of downsampling stages reuses `num_downsamples(PATCHES_PER_SIDE)` from above, so it adapts to however coarse the patch grid actually is instead of assuming a fixed depth.

Each `ConvNeXtV2Block`: depthwise `7x7` conv -> channels-first `LayerNorm2d` (a small wrapper, since `nn.LayerNorm` normalizes over the last dim and channels sit in dim 1 here) -> pointwise `1x1` expansion (~4x) -> GELU -> **Global Response Normalization** (`GRN`, Woo et al. 2023 -- the actual V2-over-V1 ingredient: each channel's global L2 norm over the spatial dims is normalized by the mean norm across channels, then used to rescale that channel, via learnable per-channel `gamma`/`beta` initialized to zero) -> pointwise `1x1` projection back down -> residual add. V2 drops V1's LayerScale in favor of GRN, per the paper. Neither `GRN` nor `LayerNorm2d` exists anywhere else in this repo today -- genuinely new code, budget some shape-checking time here specifically.

In [ ]:
class LayerNorm2d(nn.Module):
    """Channels-first LayerNorm: normalizes over the channel dim of an (B,C,H,W) tensor by
    permuting to channels-last, applying nn.LayerNorm, and permuting back."""

    def __init__(self, num_channels, eps=1e-6):
        super().__init__()
        self.ln = nn.LayerNorm(num_channels, eps=eps)

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)
        x = self.ln(x)
        return x.permute(0, 3, 1, 2)


class GRN(nn.Module):
    """Global Response Normalization (Woo et al. 2023, ConvNeXt V2) -- the V2-specific
    ingredient over V1. gamma/beta start at zero so the block starts as an identity-ish
    residual."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.zeros(1, dim, 1, 1))
        self.beta = nn.Parameter(torch.zeros(1, dim, 1, 1))

    def forward(self, x):
        gx = torch.norm(x, p=2, dim=(2, 3), keepdim=True)  # per-channel global L2 norm, (B,C,1,1)
        nx = gx / (gx.mean(dim=1, keepdim=True) + self.eps)  # normalize by cross-channel mean norm
        return x + self.gamma * (x * nx) + self.beta


class ConvNeXtV2Block(nn.Module):
    def __init__(self, dim, mlp_ratio=4):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = LayerNorm2d(dim)
        hidden = int(dim * mlp_ratio)
        self.pwconv1 = nn.Conv2d(dim, hidden, kernel_size=1)
        self.act = nn.GELU()
        self.grn = GRN(hidden)
        self.pwconv2 = nn.Conv2d(hidden, dim, kernel_size=1)

    def forward(self, x):
        residual = x
        x = self.dwconv(x)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.grn(x)
        x = self.pwconv2(x)
        return residual + x


class Downsample(nn.Module):
    """LayerNorm2d -> stride-2 conv, standard ConvNeXt downsampling between stages."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.norm = LayerNorm2d(in_channels)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=2, stride=2)

    def forward(self, x):
        return self.conv(self.norm(x))


class ConvNeXtV2(nn.Module):
    """Actuator-aligned-patch ConvNeXt V2. The shared PatchEmbed stem's output stays a 2-D
    feature map (n_channels*embed_dim_per_pupil channels at PATCHES_PER_SIDE resolution);
    num_downsamples(patches_per_side) decides how many further downsampling stages the
    patch grid can support."""

    def __init__(self, Nout, n_channels, Nmodes, patches_per_side,
                 embed_dim_per_pupil=22, blocks_per_stage=2, mlp_ratio=4):
        super().__init__()
        self.patch_embed = PatchEmbed(Nout, n_channels, embed_dim_per_pupil, patches_per_side)
        stem_channels = n_channels * embed_dim_per_pupil

        n_downsample_stages = num_downsamples(patches_per_side)
        stage_channels = [stem_channels] + [stem_channels * 2 ** (i + 1) for i in range(n_downsample_stages)]

        self.stages = nn.ModuleList([
            nn.Sequential(*[ConvNeXtV2Block(c, mlp_ratio) for _ in range(blocks_per_stage)])
            for c in stage_channels
        ])
        self.downsamples = nn.ModuleList([
            Downsample(stage_channels[i], stage_channels[i + 1]) for i in range(len(stage_channels) - 1)
        ])

        self.norm_head = LayerNorm2d(stage_channels[-1])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(stage_channels[-1], Nmodes)

    def forward(self, x):
        feat = self.patch_embed(x)
        for i, stage in enumerate(self.stages):
            feat = stage(feat)
            if i < len(self.downsamples):
                feat = self.downsamples[i](feat)
        feat = self.norm_head(feat)
        feat = self.pool(feat).flatten(1)
        return self.head(feat)

## Instantiating all three

Each architecture is built once here and printed with its total trainable parameter count. `ViT`'s `embed_dim_per_pupil=248, depth=4, n_heads=8` and `ConvNeXtV2`'s `embed_dim_per_pupil=22, blocks_per_stage=2` were hand-tuned (unlike the CNN family's single `base_channels` knob, neither family reduces to one scalar) to land within ~1% of `ClassicCNN`'s ~3.05M parameters (`base_channels=51`, at the shared `Nmodes=190`) at the default `PATCHES_PER_SIDE`/`Nout` -- so the closed-loop comparison below reflects architectural differences rather than one network having more capacity. Changing `PATCHES_PER_SIDE` changes these counts (mainly `ConvNeXtV2`'s, through its stage/downsample depth) and may need re-tuning to stay matched.

In [ ]:
# All three sized to ~3M trainable parameters at Nmodes=190 / PATCHES_PER_SIDE=11 / Nout=42:
#   Classic     base_channels=51 -> 3,054,019 (the "3M_wide" network of WidthDepthComparison.ipynb)
#   ViT         width 248, depth 4, 8 heads -> 3,031,494 (-0.7% vs Classic)
#   ConvNeXtV2  width 22 per pupil, 2 blocks per stage -> 3,062,150 (+0.3% vs Classic)
architectures = {
    "Classic": ClassicCNN(n_channels, Nmodes, Nout, base_channels=51).to(device=device),
    "ViT": ViT(Nout, n_channels, Nmodes, PATCHES_PER_SIDE,
               embed_dim_per_pupil=248, depth=4, n_heads=8).to(device=device),
    "ConvNeXtV2": ConvNeXtV2(Nout, n_channels, Nmodes, PATCHES_PER_SIDE,
                              embed_dim_per_pupil=22, blocks_per_stage=2).to(device=device),
}

for name, model in architectures.items():
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name:12s} -- total trainable parameters: {total_params:,}")

## Training

Following `CNNArchitectureComparison.ipynb`'s exact procedure: a fresh `AdamW` optimizer and a fresh `Trainer` per architecture, all sharing the same `wfs`/`dm`/`framePreprocessor`/`dataset`/`loss` objects, each calling `trainer.train(TrainRunNb, ClosedLoopIterations)`. Per the Q&A in `Ideas/11-advanced-architecture-comparison.md`, all three use the *same* optimizer/learning rate rather than per-architecture tuned recipes -- keeps architecture the only controlled variable, at the risk of understating ViT/ConvNeXt V2's potential under their own typically-recommended schedules.

Same deliberate trade-off as the six-architecture notebook: `Trainer.train()` redraws `self.dataset[0]` fresh at every outer step regardless of which `Trainer` instance calls it, so the three architectures train on independent i.i.d. draws from the same distribution rather than paired ones. Only the seeded closed-loop rollout further down is a true apples-to-apples comparison; the training curves are directionally informative only.

Self-attention's `O(N^2)` cost over `ViT`'s 484 tokens makes this a real wall-clock cost on a single memory-constrained GPU -- consider a smaller `TrainRunNb` for an initial correctness pass (confirm every shape/forward pass works, spot-check the patch-grid cell above, etc.) before committing to the full training budget below.

In [ ]:
# Closed-loop (BPTT) iterations per optimizer step in the second training stage. Stage 1 uses
# TrainParams['ClosedLoopIterations'] from CNNComparison_params.py (a single pass); this is the
# one training setting that is not in that shared file, so it lives here.
CLOSED_LOOP_ITERATIONS_STAGE2 = 5

In [ ]:
trainers = {}
loss_trackers = {}
loss_trackers_ideal = {}

for name, model in architectures.items():
    print(f"=== Training {name} (stage 1: single pass) ===")

    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['lrn'], fused=(device == 'cuda'))

    trainer = Trainer(
        wfs=wfs,
        framePreprocessor=framePreprocessor,
        dm=dm,
        M2C=M2C,
        phaseReconstructor=model,
        dataset=dataset,
        loss=loss,
        optimizer=optimizer,
    )

    try:
        trainer.load_checkpoint(PATH / f"{name}.pth", load_optimizer=False)
    except (FileNotFoundError, RuntimeError):
        # RuntimeError covers a shape mismatch against a checkpoint saved by an earlier,
        # differently-sized version of this architecture (e.g. after editing
        # PATCHES_PER_SIDE or an architecture's hyperparameters above) -- old checkpoints
        # under PATH are then no longer compatible and should be deleted or ignored.
        print("Starting from scratch")

    loss_tracker, loss_tracker_ideal = trainer.train(TrainParams['TrainRunNb'], TrainParams['ClosedLoopIterations'])

    trainer.save_checkpoint(PATH / f"{name}.pth")

    trainers[name] = trainer
    loss_trackers[name] = loss_tracker
    loss_trackers_ideal[name] = loss_tracker_ideal

### Stage 2 -- closed-loop training

The loop above (stage 1) trains every model with `TrainParams['ClosedLoopIterations']` from `CNNComparison_params.py`, i.e. a single pass per optimizer step. The next cell repeats the identical loop with `CLOSED_LOOP_ITERATIONS_STAGE2` closed-loop iterations per optimizer step, continuing from the weights stage 1 just saved (`load_checkpoint` at the top of the loop, and a fresh `AdamW` per model, as in stage 1).

The checkpoint under `PATH` is overwritten with the stage-2 weights, and from here on `trainers`/`loss_trackers`/`loss_trackers_ideal` -- the ones the plotting and evaluation cells below use -- refer to stage 2.

In [ ]:
trainers = {}
loss_trackers = {}
loss_trackers_ideal = {}

for name, model in architectures.items():
    print(f"=== Training {name} (stage 2: closed loop) ===")

    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['lrn'], fused=(device == 'cuda'))

    trainer = Trainer(
        wfs=wfs,
        framePreprocessor=framePreprocessor,
        dm=dm,
        M2C=M2C,
        phaseReconstructor=model,
        dataset=dataset,
        loss=loss,
        optimizer=optimizer,
    )

    try:
        trainer.load_checkpoint(PATH / f"{name}.pth", load_optimizer=False)
    except (FileNotFoundError, RuntimeError):
        # RuntimeError covers a shape mismatch against a checkpoint saved by an earlier,
        # differently-sized version of this architecture (e.g. after editing
        # PATCHES_PER_SIDE or an architecture's hyperparameters above) -- old checkpoints
        # under PATH are then no longer compatible and should be deleted or ignored.
        print("Starting from scratch")

    loss_tracker, loss_tracker_ideal = trainer.train(TrainParams['TrainRunNb'], CLOSED_LOOP_ITERATIONS_STAGE2)

    trainer.save_checkpoint(PATH / f"{name}.pth")

    trainers[name] = trainer
    loss_trackers[name] = loss_tracker
    loss_trackers_ideal[name] = loss_tracker_ideal

## Comparing training-loss curves

`Trainer.plot_losses` only plots one tracker pair at a time, so we reimplement just its smoothing-and-overlay logic here for all three trackers at once. The oracle "ideal loss" bound (`z_inv` applied to the true residual OPD) is a property of the atmosphere/noise distribution and the DM, not of the architecture, so the three `loss_tracker_ideal` curves are statistically equivalent -- only one is shown, as a reference lower bound shared by all three.

In [ ]:
def smooth(x, window=100):
    x = x.detach().cpu().numpy()
    return np.convolve(x, np.ones(window) / window, "valid")

fig, ax = plt.subplots(figsize=(8, 5))
for name, tracker in loss_trackers.items():
    ax.plot(smooth(tracker), label=name)

first_ideal = next(iter(loss_trackers_ideal.values()))
ax.plot(smooth(first_ideal), label="Oracle bound (ideal)", color="black", linestyle="--")

ax.set_xlabel("Iteration")
ax.set_ylabel(r"Loss")
ax.legend()
plt.show()

## Comparing closed-loop residual wavefront error (nm RMS)

For each architecture, reseed immediately before the rollout so all three see identical wind/r0/noise draws, then run `trainer.evaluate()` and compute the steady-state residual wavefront error in nm RMS over the pupil, excluding the leaky-integrator warm-up (`i > n_steps * 0.3`, the same convention `Trainer.evaluate()` and `CNNArchitectureComparison.ipynb` use).

In [ ]:
def residual_nm_rms(result, pupil, warmup_fraction=0.3):
    """Steady-state residual wavefront error (nm RMS) over the pupil, from an
    EvaluationResult's residual_opd trajectory, excluding the leaky-integrator warm-up
    period (same 0.3 convention Trainer.evaluate() uses internally)."""
    n_steps = result.residual_opd.shape[0]
    warmup = int(n_steps * warmup_fraction)
    steady_state = result.residual_opd[warmup:]
    pupil_values = steady_state[..., pupil.bool()]
    return torch.sqrt(torch.mean(pupil_values ** 2)).item() * 1e9


eval_dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
eval_dataset.generateClosedLoop = True

nm_rms = {}
for name, trainer in trainers.items():
    torch.manual_seed(TrainParams['Seed'])
    result = trainer.evaluate(n_steps=TrainParams['TestRunNb'], dataset=eval_dataset)
    nm_rms[name] = residual_nm_rms(result, dataset.pupil)
    print(f"{name:12s} -- steady-state residual: {nm_rms[name]:.1f} nm RMS")

In [ ]:
names = list(nm_rms.keys())
rms_values = [nm_rms[n] for n in names]
param_counts = [sum(p.numel() for p in architectures[n].parameters() if p.requires_grad) for n in names]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].bar(names, rms_values)
axes[0].set_ylabel("Steady-state residual (nm RMS)")
axes[0].set_title("Closed-loop performance")
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(names, param_counts)
axes[1].set_ylabel("Trainable parameters")
axes[1].set_title("Model size")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## Visualizing the best architecture's closed loop

A closer look at whichever architecture came out on top above -- the same rollout-animation pattern `CNNArchitectureComparison.ipynb`/`05_TrainingAReconstructor.ipynb` use.

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

best_name = min(nm_rms, key=nm_rms.get)
print(f"Best architecture: {best_name} ({nm_rms[best_name]:.1f} nm RMS)")

n_frames = 60
torch.manual_seed(TrainParams['Seed'])
result = trainers[best_name].evaluate(n_steps=n_frames, dataset=eval_dataset)

fig, axes = imshow_multiple(
    [
        {"tensor": result.opd[0], "title": "Input OPD", "same_scale": True},
        {"tensor": result.residual_opd[0], "title": "Residual OPD", "scale_reference": result.opd[0]},
        {"tensor": result.wfs_frames[0], "title": "WFS frame"},
        {"tensor": torch.sqrt(result.psfs[0]), "title": "PSF", "same_scale": True},
    ],
    max_channel_number=9
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result.opd[i], "title": "Input OPD", "same_scale": True},
            {"tensor": result.residual_opd[i], "title": "Residual OPD", "scale_reference": result.opd[i]},
            {"tensor": result.wfs_frames[i], "title": "WFS frame"},
            {"tensor": torch.sqrt(result.psfs[i]), "title": "PSF", "same_scale": True},
        ],
        fig=fig, axes=axes,
        max_channel_number=9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())